# 01 · 20261027 从 OpenAI SDK 到 LangChain 服务化

> 课堂练习：`20261027/` 下 9 个脚本 + 共用的 `lab_config.py`。
> 本笔记按学习顺序串起来，并把脚本里的硬编码 Key / 局域网 IP 统一改成读 `week23/.env`。

## 今日主线

**直连大模型** → **三种建模型方式** → **消息与多轮对话** → **FastAPI 服务化** → **请求解析工程化**

## 文件 ↔ 主题对照

| 脚本 | 主题 | 关键点 |
|---|---|---|
| `01_openai_sdk_stream` | OpenAI SDK 直连 + 流式 | `chat.completions.create(stream=True)`、`reasoning_content` |
| `02_model_init_three_ways` | 三种建模型方式 | `ChatOpenAI` / `ChatOllama` / `init_chat_model` |
| `03_json_output` | 让模型输出 JSON | 靠 system 提示词约束格式 |
| `04_message_types` | LangChain 消息类型 | `SystemMessage` / `HumanMessage` / `AIMessage` |
| `05_multi_turn_cli` | 多轮对话 CLI | 历史列表不断追加 |
| `06_chat_server_fastapi` | 模型包装成接口 | `POST /chat` + 重试 + 备用模型 |
| `07_chat_client` | 客户端调用 | `requests` + 多轮 |
| `08_fastapi_request_basics` | 请求解析工程化 | body / headers / query、中间件、统一异常 |
| `09_requests_params` | 参数该放哪 | `params` / `headers` / `json` |
| `lab_config` | 统一配置 | `.env` + `get_model()` |

## 一句话总结

01~05 解决「**怎么把模型接进来**」，06~09 解决「**怎么把模型服务出去**」。

## 0 · 统一配置

课堂代码里散落着 `sk-ws-*` 的 Key 和老师机器的局域网地址（Ollama `192.168.1.105:11434`）。这里全部收口到 `lab_config.py`：

- 模型来源由 `.env` 的 `LLM_PROVIDER` 决定（`qwen` / `openai` / `ollama`）
- `get_model()` 统一构造 ChatModel：云端走 OpenAI 兼容协议，本地走 `ChatOllama`（并默认关掉 thinking）
- 千问是思考型模型，`QWEN_ENABLE_THINKING=false` 可避免 reasoning token 把 `max_tokens` 吃光

下面这格只读配置，**不联网、不计费**。

In [1]:
from pathlib import Path
import sys


def find_lab_dir(start: Path) -> Path:
    """向上找含 lab_config.py 的目录（20261027/），复用 week23 的统一配置。"""
    for base in [start, *start.parents]:
        for candidate in (base, base / "20261027"):
            if (candidate / "lab_config.py").exists():
                return candidate
    raise FileNotFoundError("没找到 lab_config.py，请在 week23 目录下打开本笔记本")


sys.path.insert(0, str(find_lab_dir(Path.cwd())))

from lab_config import (
    API_KEY,
    BASE_URL,
    MAX_TOKENS,
    MODEL,
    PROVIDER,
    get_model,
    show_config,
)

show_config()

provider = qwen
model    = qwen3.8-flash
base_url = https://cn-beijing.maas.qianwenaiapi.com/compatible-mode/v1
api_key  = sk-ws-***ef-t
max_tokens = 1024
thinking   = False（true 会白烧 reasoning token）


In [2]:
# 会真正调用模型的单元格都套了这个开关：默认 False，避免误执行烧 token
RUN_DEMOS = False

## 1 · 直接用 OpenAI SDK 调模型（流式 + 思考过程）

访问线上大模型 = **地址 + Key + 模型名**，三样齐了就能发请求。

LangChain 只是封装，底层仍然是 OpenAI 那套 HTTP 协议，所以先看懂最原始的调用：

| 参数 | 作用 |
|---|---|
| `messages=[{"role": "user", "content": ...}]` | 对话内容，role 决定谁在说话 |
| `stream=True` | 不一次性等完整回复，而是边生成边返回 chunk |
| `extra_body={"enable_thinking": True}` | 非标准字段，千问网关用它开关思考链 |

流式返回的是一个**生成器**，每个 chunk 的结构：

```text
ChatCompletionChunk(
    choices=[Choice(delta=ChoiceDelta(reasoning_content="用户问的是…", content=None))],
    ...
)
```

- `delta.reasoning_content`：思考链（千问、DeepSeek-R1 这类模型才有）
- `delta.content`：正式回答
- 有些 chunk 的 `choices` 是空列表（例如最后的 usage 统计），要先跳过

In [3]:
from openai import OpenAI

if RUN_DEMOS:
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "你是谁"}],
        extra_body={"enable_thinking": True},   # 只有千问网关认这个字段
        stream=True,
    )

    answering = False      # ← 原脚本写成 True，导致思考过程永远不打印
    print("=" * 20 + "思考过程" + "=" * 20)
    for chunk in completion:
        if not chunk.choices:                  # 收尾的 usage chunk
            continue
        delta = chunk.choices[0].delta

        if getattr(delta, "reasoning_content", None):
            if not answering:
                print(delta.reasoning_content, end="", flush=True)
        if delta.content:
            if not answering:
                print("\n" + "=" * 20 + "完整回复" + "=" * 20)
                answering = True
            print(delta.content, end="", flush=True)
else:
    print("（RUN_DEMOS=False，跳过流式调用。原脚本见 01_openai_sdk_stream.py）")

（RUN_DEMOS=False，跳过流式调用。原脚本见 01_openai_sdk_stream.py）


### 1.1 原脚本的一个小 bug

```python
is_answering = True          # ← 初始值反了
...
if delta.reasoning_content is not None:
    if not is_answering:     # 永远不成立
        print(delta.reasoning_content, end="")
```

因为 `is_answering` 一开始就是 `True`，`if not is_answering` 一直为假：

- 「思考过程」标题打印了，但思考内容一段都没输出
- 进入正文时也检测不到「从思考切到回答」这个转折，标题「完整回复」永远不会出现

**正确做法**：初始值为 `False`，等看到第一个 `content` 再翻转，本笔记已修正。

## 2 · 三种创建模型的方式

| 方式 | 适用 | 关键差异 |
|---|---|---|
| `ChatOpenAI` | 任意 OpenAI 兼容云端（DeepSeek / 百炼 / 智谱…） | 参数直接用 `max_tokens` |
| `ChatOllama` | 本地 Ollama | 走原生 `/api/chat`，长度参数叫 `num_predict`；`reasoning=False` 可关思考 |
| `init_chat_model` | 想用一个工厂切换 provider | 传 `model_provider="openai"` 即可，换服务只改配置 |

三者都返回 `AIMessage`，都能 `.invoke(messages)` —— 接口一致，这是 LangChain 的核心价值。

In [4]:
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain.chat_models import init_chat_model

messages = [{"role": "user", "content": "你是谁"}]

if RUN_DEMOS:
    # 1) 云端：ChatOpenAI（底层就是 OpenAI 协议）
    m1 = ChatOpenAI(model=MODEL, base_url=BASE_URL, api_key=API_KEY, max_tokens=MAX_TOKENS)
    print("ChatOpenAI      :", m1.invoke(messages).content[:60])

    # 2) 本地：ChatOllama 走原生接口（长度参数是 num_predict，不是 max_tokens）
    # m2 = ChatOllama(model="qwen3.5:2b", base_url="http://127.0.0.1:11434",
    #                 num_predict=MAX_TOKENS, reasoning=False)
    # print("ChatOllama      :", m2.invoke(messages).content[:60])
    print("ChatOllama      : （本机 Ollama 无模型，跳过）")

    # 3) 统一工厂：换 provider 只改参数
    m3 = init_chat_model(
        model=MODEL, base_url=BASE_URL, api_key=API_KEY,
        max_tokens=MAX_TOKENS, model_provider="openai",
    )
    print("init_chat_model :", m3.invoke(messages).content[:60])
else:
    print("（RUN_DEMOS=False，跳过。当前 PROVIDER =", PROVIDER, "）")

（RUN_DEMOS=False，跳过。当前 PROVIDER = qwen ）


## 3 · 让模型输出 JSON

模型本身不会「返回对象」，只能返回文本。要稳定拿到 JSON，两步：

1. **system 里写清格式**：`你必须以 json 格式输出，格式如下：{"title": ..., "content": ...}`
2. **拿回来自己解析**：`json.loads(response.content)`（更稳的做法见 20261029 的 `JsonOutputParser` / `PydanticOutputParser`）

### ⚠️ 原脚本的坑

```python
{"role": "user", "content": "模仿朝花夕拾的风格产出文章……文章是：{article}"}
```

这是**普通字符串**，没有 `f` 前缀也没有 `.format()`，`{article}` 会原样发给模型 —— 那一大段《朝花夕拾》正文根本没参与推理。

本笔记改成 f-string，并换成长度可控的短片段，避免一次请求几千 token。

In [5]:
import json

# 只截取一小段作为风格参考（原文近两千字，按 token 计费不划算）
article = "我常想在纷扰中寻出一点闲静来，然而委实不容易。目前是这么离奇，心里是这么芜杂。……"

messages = [
    {
        "role": "system",
        "content": (
            "你是一名文学家，擅长模仿别人的风格写文章。"
            '必须以 JSON 输出，格式：{"title": 文章标题, "content": 文章内容}'
        ),
    },
    {
        "role": "user",
        "content": f"模仿《朝花夕拾》的风格写一篇文章，主题：如何提高写作能力。参考文章：{article}",
    },
]

if RUN_DEMOS:
    response = get_model(temperature=1.2, top_p=0.9).invoke(messages)
    print(type(response).__name__)       # AIMessage，不是 str
    print(response.content)
    try:
        data = json.loads(response.content)
        print("解析成功:", list(data))
    except json.JSONDecodeError:
        print("模型没吐标准 JSON，需要 Prompt 约束或 OutputParser")
else:
    print("（RUN_DEMOS=False，跳过）待发送的 user 内容：\n", messages[1]["content"])

（RUN_DEMOS=False，跳过）待发送的 user 内容：
 模仿《朝花夕拾》的风格写一篇文章，主题：如何提高写作能力。参考文章：我常想在纷扰中寻出一点闲静来，然而委实不容易。目前是这么离奇，心里是这么芜杂。……


## 4 · LangChain 的消息类型

除了 `{"role": ..., "content": ...}` 字典，LangChain 还有对应的**消息类**，字段更丰富（还能带 `tool_calls`、`response_metadata` 等）：

| 类 | `type` | 用途 |
|---|---|---|
| `SystemMessage` | `system` | 人设 / 规则 / 输出格式约束 |
| `HumanMessage` | `human` | 用户输入 |
| `AIMessage` | `ai` | 模型回复，可直接追加进历史 |
| `ToolMessage` | `tool` | 工具调用结果（Agent 章节会用到） |

两种写法可以混用，LangChain 会自动把字典转成对应的消息对象。

In [7]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage

system_message = SystemMessage(content="你是文学家，必须以 JSON 输出")
human_message = HumanMessage(content="你是谁")
ai_message = AIMessage(content="我是千问")
new_message = HumanMessage(content="模仿朝花夕拾的风格产出文章")

messages = [system_message, human_message, ai_message, new_message]

for m in messages:
    print(f"{type(m).__name__:15s} type={m.type:6s} content={m.content[:16]}")

print()
print("字典的 role 与消息对象的 type 对应关系（注意 user → human）：")
print('  {"role": "system"}    -> type =', SystemMessage(content="").type)
print('  {"role": "user"}      -> type =', HumanMessage(content="").type)
print('  {"role": "assistant"} -> type =', AIMessage(content="").type)

SystemMessage   type=system content=你是文学家，必须以 JSON 输
HumanMessage    type=human  content=你是谁
AIMessage       type=ai     content=我是千问
HumanMessage    type=human  content=模仿朝花夕拾的风格产出文章

字典的 role 与消息对象的 type 对应关系（注意 user → human）：
  {"role": "system"}    -> type = system
  {"role": "user"}      -> type = human
  {"role": "assistant"} -> type = ai


## 5 · 多轮对话：把历史全部带上

大模型是**无状态**的：它不会记得上一轮。所谓「多轮」，就是每次请求都把之前的对话重新发一遍。

```text
第 1 轮 发送：[system, user1]                    → 得到 ai1
第 2 轮 发送：[system, user1, ai1, user2]        → 得到 ai2
第 3 轮 发送：[system, user1, ai1, user2, ai2, user3] → 得到 ai3
```

所以只要 `messages` 列表一直追加，模型就「有记忆」了。两个细节：

- 模型的回复是 `AIMessage` 对象，可以直接 `append` 进列表（不用手动转成 dict）
- CLI 里用「空回车退出」而不是死循环，避免失控烧 token

> 20261029 学到的 `RunnableWithMessageHistory` 就是把这套「手动追加」自动化，还能按 `session_id` 隔离。

In [8]:
def multi_turn_conversation(messages):
    """多轮对话：每轮把用户输入和模型回复都追加进历史。

    注意：这里只定义不调用（input() 会阻塞 notebook 内核）。
    实际使用见 05_multi_turn_cli.py。
    """
    model = get_model(temperature=1.2)
    while True:
        question = input("请输入你的问题（空回车退出）：").strip()
        if not question:
            break
        messages.append({"role": "user", "content": question})
        response = model.invoke(messages)
        print(response.content)
        messages.append(response)          # AIMessage 直接追加
        print("-" * 100)


print("multi_turn_conversation 已定义；每轮发送的消息条数 = 1(system) + 2×轮数")

multi_turn_conversation 已定义；每轮发送的消息条数 = 1(system) + 2×轮数


## 6 · 服务化：把模型包装成 HTTP 接口

`06_chat_server_fastapi.py` 做三件事：**接请求 → 调模型（带兜底）→ 返回 JSON**。

```mermaid
graph LR
    C[客户端 07] -->|POST /chat<br/>messages| S[FastAPI 06]
    S --> R{主模型}
    R -->|成功| A[返回 ai_content]
    R -->|失败·重试 3 次| R
    R -->|仍失败| B[备用模型 DeepSeek]
    B --> A
    R -->|全挂| E[502 错误]
```

课堂原版（`teste6.py`）的三个问题，本地化版都修掉了：

| # | 问题 | 后果 |
|---|---|---|
| 1 | 用 `response.status_code` 判断成败 | `AIMessage` 没这个属性 → `AttributeError` |
| 2 | 没有 `try/except` | 网络异常直接抛出，备用模型永远不会被调用 |
| 3 | 循环 3 次后 `response` 可能还是 `None` | 接口返回 `{"ai_content": null}`，客户端拿到空值 |

另外接口名从 `/users` 改成 `/chat`，只监听 `127.0.0.1`（本机自用，不对外网开放）。

下面用**假模型**演示重试 + 兜底逻辑，完全不联网。

In [9]:
class FakeModel:
    """假模型：前 fail_times 次抛异常，之后返回固定内容。"""

    def __init__(self, name: str, fail_times: int = 0):
        self.name = name
        self.fail_times = fail_times
        self.calls = 0

    def invoke(self, messages):
        self.calls += 1
        if self.calls <= self.fail_times:
            raise RuntimeError(f"{self.name} 第 {self.calls} 次调用失败（模拟网络超时）")
        return AIMessage(content=f"{self.name} 的回复（第 {self.calls} 次调用）")


RETRIES = 3


def invoke_with_retry(messages, primary, backup=None):
    """先重试主模型，仍失败则切备用模型；全挂了抛异常。"""
    last_error = None

    for attempt in range(1, RETRIES + 1):
        try:
            resp = primary.invoke(messages)
            if resp.content:
                print(f"  主模型成功（第 {attempt} 次尝试）")
                return resp.content
            print(f"  主模型返回空内容（第 {attempt} 次尝试）")
        except Exception as exc:
            last_error = exc
            print(f"  主模型失败（第 {attempt} 次尝试）：{exc}")

    if backup is not None:
        try:
            print("  主模型重试耗尽 → 切备用模型")
            resp = backup.invoke(messages)
            if resp.content:
                return resp.content
        except Exception as exc:
            last_error = exc

    raise RuntimeError(f"模型调用失败：{last_error}")


msgs = [{"role": "user", "content": "hi"}]

print("场景 A：主模型第 2 次成功")
print("  ->", invoke_with_retry(msgs, FakeModel("主模型", fail_times=1)), "\n")

print("场景 B：主模型全挂，备用模型兜底")
print("  ->", invoke_with_retry(msgs, FakeModel("主模型", fail_times=99), FakeModel("备用模型")), "\n")

print("场景 C：主备都挂")
try:
    invoke_with_retry(msgs, FakeModel("主模型", fail_times=99), FakeModel("备用", fail_times=99))
except RuntimeError as exc:
    print("  FastAPI 里这会变成 HTTPException(502)：", exc)

场景 A：主模型第 2 次成功
  主模型失败（第 1 次尝试）：主模型 第 1 次调用失败（模拟网络超时）
  主模型成功（第 2 次尝试）
  -> 主模型 的回复（第 2 次调用） 

场景 B：主模型全挂，备用模型兜底
  主模型失败（第 1 次尝试）：主模型 第 1 次调用失败（模拟网络超时）
  主模型失败（第 2 次尝试）：主模型 第 2 次调用失败（模拟网络超时）
  主模型失败（第 3 次尝试）：主模型 第 3 次调用失败（模拟网络超时）
  主模型重试耗尽 → 切备用模型
  -> 备用模型 的回复（第 1 次调用） 

场景 C：主备都挂
  主模型失败（第 1 次尝试）：主模型 第 1 次调用失败（模拟网络超时）
  主模型失败（第 2 次尝试）：主模型 第 2 次调用失败（模拟网络超时）
  主模型失败（第 3 次尝试）：主模型 第 3 次调用失败（模拟网络超时）
  主模型重试耗尽 → 切备用模型
  FastAPI 里这会变成 HTTPException(502)： 模型调用失败：备用 第 1 次调用失败（模拟网络超时）


## 7 · 客户端：用 requests 调自己的服务

服务跑在 `http://127.0.0.1:8000/chat`，请求体是 `{"messages": [...]}`，返回 `{"ai_content": "..."}`。

客户端自己维护历史，每轮把完整 `messages` 发过去 —— 和服务端「无状态」的分工就清楚了。

### ⚠️ 原脚本的 bug

```python
response = requests.post(url, json=data)        # data 是全局变量
```

形参 `messages` 成了摆设：用户在 CLI 里输入的内容追加到了 `messages`，发出去的却是最初的 `data`。
本笔记改成 `json={"messages": messages}`。

In [10]:
import requests

CHAT_URL = "http://127.0.0.1:8000/chat"

if RUN_DEMOS:
    # 前提：另开终端 `python 06_chat_server_fastapi.py`
    resp = requests.post(
        CHAT_URL,
        json={"messages": [{"role": "user", "content": "你是谁"}]},
        timeout=120,
    )
    print(resp.status_code, resp.json())
else:
    print("（RUN_DEMOS=False，跳过。需先启动 06_chat_server_fastapi.py）")

（RUN_DEMOS=False，跳过。需先启动 06_chat_server_fastapi.py）


## 8 · 请求解析：数据到底放在哪

一个 HTTP 请求有三处能放数据，服务端取值方式各不相同：

| 位置 | 客户端写法 | 服务端取值 | 典型用途 |
|---|---|---|---|
| 路径 / 查询参数 | `requests.get(url, params={...})` | `request.query_params.get(...)` | 过滤、分页、`operatorId` |
| 请求头 | `requests.get(url, headers={...})` | `request.headers.get(...)` | Token、鉴权、Content-Type |
| 请求体 | `requests.post(url, json={...})` | `await request.json()` | 业务数据 |

真实例子（钉钉待办接口）：

```text
POST /v1.0/todo/users/{id}/tasks?operatorId=PUoiinWIpxxx
x-acs-dingtalk-access-token: xxxxx          ← 请求头
Content-Type: application/json              ← 请求头
{"sourceId": "isv_dingtalkTodo1", ...}      ← 请求体
```

`08` 在这个基础上补齐了工程必需的三样：**请求日志中间件**（request-id + 耗时）、**统一异常出口**（400/401/422/500 都是结构化 JSON）、**Pydantic 模型校验**。下面直接跑一个精简版验证。

In [11]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from starlette.exceptions import HTTPException as StarletteHTTPException

app = FastAPI(title="08 精简版")


async def read_json_or_400(request: Request) -> dict:
    """非法 JSON 是客户端问题 → 400，不是 500。"""
    try:
        data = await request.json()
    except Exception:
        raise HTTPException(status_code=400, detail="请求体不是合法 JSON") from None
    if not isinstance(data, dict):
        raise HTTPException(status_code=400, detail="请求体必须是 JSON 对象")
    return data


@app.exception_handler(StarletteHTTPException)
async def handle_http_exception(request: Request, exc: StarletteHTTPException):
    """统一异常出口：业务异常也返回结构化 JSON。"""
    return JSONResponse(status_code=exc.status_code, content={"error": exc.detail})


@app.post("/add_one")
async def add_one(request: Request):
    data = await read_json_or_400(request)
    number = data.get("number")
    token = request.headers.get("x-acs-dingtalk-access-token")
    operator_id = request.query_params.get("operatorId")

    if number is None:
        raise HTTPException(status_code=400, detail="缺少 number")
    if not token:
        raise HTTPException(status_code=401, detail="缺少 x-acs-dingtalk-access-token")

    return {"result": number + 1, "operatorId": operator_id}


class AddOneRequest(BaseModel):
    number: float = Field(..., description="要加一的数字")


@app.post("/add_one_v2")
async def add_one_v2(payload: AddOneRequest):
    """声明式校验：少字段 / 类型不对 → 框架自动 422。"""
    return {"result": payload.number + 1}


@app.post("/bad")
async def bad():
    # 课堂版写法：Flask 风格 return (dict, 400) —— 下一格验证它的实际结果
    return {"error": "缺少 number"}, 400

In [12]:
from fastapi.testclient import TestClient

client = TestClient(app)

ok = client.post(
    "/add_one",
    params={"operatorId": "PUoiinWIpxxx"},
    headers={"x-acs-dingtalk-access-token": "xxxxx"},
    json={"number": 41},
)
print("正常请求   :", ok.status_code, ok.json())

print("缺 number  :", (r := client.post("/add_one", headers={"x-acs-dingtalk-access-token": "t"}, json={})).status_code, r.json())
print("坏 JSON    :", (r := client.post("/add_one", content=b"not-json", headers={"x-acs-dingtalk-access-token": "t"})).status_code, r.json())
print("缺 token   :", (r := client.post("/add_one", json={"number": 1})).status_code, r.json())
print("Pydantic  :", (r := client.post("/add_one_v2", json={"number": "abc"})).status_code, r.json()["detail"][0]["type"])
print("return 元组:", (r := client.post("/bad")).status_code, r.json(), "← 状态码没生效！")

正常请求   : 200 {'result': 42, 'operatorId': 'PUoiinWIpxxx'}
缺 number  : 400 {'error': '缺少 number'}
坏 JSON    : 400 {'error': '请求体不是合法 JSON'}
缺 token   : 401 {'error': '缺少 x-acs-dingtalk-access-token'}
Pydantic  : 422 float_parsing
return 元组: 200 [{'error': '缺少 number'}, 400] ← 状态码没生效！


### 8.1 两个必踩的坑

**坑 1：`return {"error": ...}, 400` 不会返回 400**

这是 Flask 的写法。FastAPI 会把整个元组当**响应体**序列化：

```text
实际响应：200 OK
[{"error": "缺少 number"}, 400]
```

状态码根本没变，客户端 `raise_for_status()` 也不会报错。**正确做法是 `raise HTTPException(status_code=400, detail=...)`**，上一格 `/bad` 就是反例。

**坑 2：默认 404 的格式和业务错误不一致**

不注册 `StarletteHTTPException` 处理器时，404 返回 `{"detail": "Not Found"}`，而业务错误返回 `{"error": ...}`。
注册后（`fastapi.HTTPException` 是它的子类）所有 HTTP 错误都走同一个出口，客户端只需要认一种格式。

补充：`request.state.request_id` 由中间件写入，同一个请求内所有处理器读到的都是同一个 id，出错时能和日志对上。

## 9 · 今日踩坑清单

| # | 位置 | 问题 | 修正 |
|---|---|---|---|
| 1 | 01 流式 | `is_answering = True` 初始值反了 | 改成 `False`，看到第一个 `content` 再翻转 |
| 2 | 01 流式 | 不判断 `chunk.choices` 是否为空 | 收尾的 usage chunk 会 `IndexError` |
| 3 | 03 JSON | `"……{article}"` 是普通字符串，没替换 | 用 f-string 或 `.format()` |
| 4 | 03 JSON | `model.invoke()` 返回 `AIMessage` | 取 `.content`，再 `json.loads` |
| 5 | 02 建模型 | Ollama 原生接口用 `num_predict` | 不是 `max_tokens` |
| 6 | 02 建模型 | `qwen3.8-*` 默认开思考，吃掉额度 | `.env` 设 `QWEN_ENABLE_THINKING=false` |
| 7 | 06 服务 | `AIMessage.status_code` 不存在 | 用 `try/except` + 空内容判断 |
| 8 | 06 服务 | 没兜底，异常直接抛出 | 重试 3 次 → 备用模型 → 502 |
| 9 | 07 客户端 | `requests.post(url, json=data)` 用了全局变量 | `json={"messages": messages}` |
| 10 | 08 FastAPI | `return (dict, 400)` → 200 | `raise HTTPException(...)` |
| 11 | 05 记忆 | `while True: input()` 可能无限循环 | 空回车退出，随时能停 |
| 12 | 全部 | Key / 局域网 IP 写死在代码里 | 走 `lab_config` + `.env`，不提交到 Git |

## 小结

20261027 解决的是「**怎么把模型接进来、再服务出去**」：

```mermaid
graph LR
    A[OpenAI SDK<br/>直连 + 流式] --> B[三种建模型方式]
    B --> C[JSON 输出]
    C --> D[消息类型]
    D --> E[多轮对话]
    E --> F[FastAPI 服务化]
    F --> G[客户端调用]
    F --> H[请求解析工程化]
```

关键认知：

- **LangChain 不改变底层协议**，只是把 `ChatOpenAI` / `ChatOllama` / 工厂函数抽象成同一套 `invoke` 接口
- **模型无状态**，多轮对话全靠客户端把历史带全
- **服务端要做的事情比「调模型」多得多**：超时重试、备用模型、参数校验、统一错误格式、请求追踪

**下一步**：`20261029` 的提示词模板、Few-shot、LCEL、结构化输出与记忆，正好接上这里的消息类型和 JSON 输出。

> 配套笔记：`20261029_langchain_prompt_notes.ipynb`（同在 `week23/` 目录）